# Otimização e Comparação de Modelos de Classificação Supervisionada

Projeto de classificação para prever a adesão dos clientes da operadora Megaline ao plano Ultra a partir do comportamento de uso, com número de chamadas, minutos, mensagens e volume de dados consumidos.

Três algoritmos são comparados, Regressão Logística, Árvore de Decisão e Floresta Aleatória, com os dados divididos em treino, validação e teste. A validação é usada para o ajuste de hiperparâmetros e a escolha do melhor modelo, e o conjunto de teste fica reservado para a avaliação final.

In [77]:
# Importando blbliotecas relevantes
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [78]:
# Importando os dados
user_behavior = pd.read_csv('datasets/users_behavior.csv')

In [79]:
# Analisando os dados
print(user_behavior.head(10))
print(user_behavior.info())

   calls  minutes  messages   mb_used  is_ultra
0   40.0   311.90      83.0  19915.42         0
1   85.0   516.75      56.0  22696.96         0
2   77.0   467.66      86.0  21060.45         0
3  106.0   745.53      81.0   8437.39         1
4   66.0   418.74       1.0  14502.75         0
5   58.0   344.56      21.0  15823.37         0
6   57.0   431.64      20.0   3738.90         1
7   15.0   132.40       6.0  21911.60         0
8    7.0    43.39       3.0   2538.67         1
9   90.0   665.41      38.0  17358.61         0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB
None


O dataset user_behavior contém 5 colunas e 3214 linhas. As colunas calls minutes messages e mb_used são floats, enquanto is_ultra é int. O tipo das colunas de mensagens e chamadas ('calls' e 'messages') representam número de chamadas e número de mensagens, então o correto seriam estar como int. Todas as colunas tem 3214 valores não nulos, então não temos valores ausentes.

In [80]:
# Convertendo o tipo das colunas 'calls' e 'messages'
user_behavior['calls'] = user_behavior['calls'].astype('int')
user_behavior['messages'] = user_behavior['messages'].astype('int')
# Verificando o user_behavior
user_behavior.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   int64  
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   int64  
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(2), int64(3)
memory usage: 125.7 KB


In [81]:
# Verificando duplicados
user_behavior.duplicated().sum()

0

Não temos valores duplicados no dataframe user_behavior

# Teste de Modelos

Para avaliar os melhores resultados testarei 3 modelos diferentes de classificação:
- Regressão Logística
- Árvore de decisão
- Floresta Aleatória

Para avaliar os modelos, optei por dividir os dados na proporção de 60% para treino, 20% para validação e 20% para teste. Essa estrutura permite utilizar o conjunto de treino para ajustar os algoritmos, o conjunto de validação para a escolha dos melhores hiperparâmetros e, por fim, o conjunto de teste para a avaliação final de generalização, garantindo a integridade da análise.

In [82]:
# Definindo variáveis de entrada e alvo
features = user_behavior.drop(['is_ultra'], axis = 1)
target = user_behavior['is_ultra']

In [83]:
# Separando 20% para o conjunto de Teste Final
features_df, features_test, target_df, target_test = train_test_split(
    features, target, test_size=0.20, random_state=54321)

# Dividindo o 80% restantes, para Validação e treino
features_train, features_valid, target_train, target_valid = train_test_split(
    features_df, target_df, test_size=0.25, random_state=54321)

### Regressão Logística

In [84]:
# Criando modelo de regressão logística
model = LogisticRegression(random_state=54321, solver='liblinear')
# Treinando o modelo
model.fit(features_train, target_train)
# Obtendo a acurácia
RL_acc = model.score(features_valid, target_valid)
print('Acurácia: ', RL_acc)

Acurácia:  0.7325038880248833


### Árvore de decisão

In [85]:
# Criando modelo de Árvore de decisão, com loop para diferentes profundidades.
best_score_arv = 0
best_depth_arv = 0

for depth in range(1, 11):
    # Criando o modelo
    model = DecisionTreeClassifier(max_depth = depth, random_state=54321)
    # Treinando o modelo
    model.fit(features_train, target_train) 
    # Obtendo a acurácia
    score_arv = model.score(features_valid, target_valid)
    # Encontrando a melhor profundidade (depth)
    if score_arv > best_score_arv:
        best_score_arv = score_arv
        best_depth_arv = depth
print('A melhor acurácia do modelo teve profundidade', best_depth_arv,'com acurácia', best_score_arv)

A melhor acurácia do modelo teve profundidade 5 com acurácia 0.8180404354587869


### Floresta Aleatória

In [86]:
best_score_fl = 0
best_depth_fl = 0
best_est = 0
for depth in range (1,11):
    for est in range(1, 21):
        model = RandomForestClassifier(max_depth = depth, n_estimators= est, random_state=54321) # defina o número de árvores
        # Treinando o modelo
        model.fit(features_train, target_train)
        # Obtendo a acurácia
        score_fl = model.score(features_valid, target_valid)
        # Encontrando a melhor profundidade
        if score_fl > best_score_fl:
            best_score_fl = score_fl 
            best_depth_fl = depth
            best_est = est 
print('A melhor acurácia do modelo teve', best_est, 'árvores, com profundidade', best_depth_fl,'e acurácia', best_score_fl)

A melhor acurácia do modelo teve 5 árvores, com profundidade 9 e acurácia 0.8460342146189735


In [87]:
# Treinando o modelo campeão (Floresta Aleatória) com a união dos conjuntos de treino e validação
final_model = RandomForestClassifier(max_depth=best_depth_fl, n_estimators=best_est, random_state=54321)
final_model.fit(features_df, target_df)

# Avaliação final no conjunto de teste 
test_acc = final_model.score(features_test, target_test)
print(f'Acurácia final no conjunto de teste: {test_acc:.4f}')

Acurácia final no conjunto de teste: 0.7792


# Conclusão e Análise dos Resultados

### 1. Preparação e Pré-processamento dos Dados
A base de dados `user_behavior.csv`, composta por 3.214 registros e 5 colunas, passou por uma análise inicial de qualidade. Não foram identificados valores ausentes nem registros duplicados. Para garantir a coerência com as variáveis, as colunas `calls` e `messages` foram convertidas do tipo `float` para `int`, visto que representam contagens de chamadas e mensagens.

### 2. Estratégia de Divisão do Dataset
Para garantir a confiabilidade da avaliação, o dataset foi dividido em três conjuntos distintos na proporção de **60% para treino, 20% para validação e 20% para teste**:
- **Conjunto de Treino (60%):** Utilizado para o aprendizado dos modelos.
- **Conjunto de Validação (20%):** Utilizado para a otimização de hiperparâmetros e seleção do melhor modelo.
- **Conjunto de Teste (20%):** Mantido isolado para a avaliação final com dados inéditos.

### 3. Comparação de Modelos e Otimização
Três algoritmos de classificação foram avaliados e comparados no conjunto de validação:
- **Regressão Logística:** Atingiu uma acurácia de **73,25%**.
- **Árvore de Decisão:** Otimizada via variação de profundidade (`max_depth`), obtendo melhor desempenho com **profundidade 5** e acurácia de **81,80%**.
- **Floresta Aleatória:** Otimizada variando profundidade (`max_depth`) e número de estimadores (`n_estimators`), obtendo a melhor performance no conjunto de validação com **5 árvores e profundidade 9**, alcançando acurácia de **84,60%**.

### 4. Avaliação Final
O modelo selecionado como campeão na fase de validação foi a **Floresta Aleatória**. Para maximizar o aprendizado antes da avaliação definitiva, o modelo foi retreinado combinando os conjuntos de treino e validação (80% dos dados) com os hiperparâmetros otimizados (`max_depth=9` e `n_estimators=5`). 

Na avaliação final sobre o conjunto de teste (20%), o modelo alcançou uma acurácia de **77,92%**, confirmando sua capacidade de generalização para dados não vistos.